In [3]:
# =====================================
# IMPORT LIBRARIES
# =====================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
from sklearn.svm import SVC

# =====================================
# LOAD DATASET
# =====================================

df = pd.read_csv("AI_Resume_Screening.csv")

print("Dataset Shape:", df.shape)
print(df.head())

# =====================================
# DROP UNNECESSARY COLUMNS
# =====================================

drop_cols = ["Resume_ID", "Name"]

for col in drop_cols:
    if col in df.columns:
        df.drop(col, axis=1, inplace=True)

# =====================================
# HANDLE MISSING VALUES
# =====================================

df["Certifications"] = df["Certifications"].fillna("Not Available")

# =====================================
# TARGET VARIABLE
# =====================================

target_column = "Recruiter Decision"

X = df.drop(target_column, axis=1)
y = df[target_column]

# =====================================
# IDENTIFY COLUMN TYPES
# =====================================

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical Columns:")
print(categorical_cols)

print("\nNumerical Columns:")
print(numerical_cols)

# =====================================
# PREPROCESSING
# =====================================

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

# =====================================
# TRAIN TEST SPLIT
# =====================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =====================================
# MODELS
# =====================================

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(),
    "SVM": SVC(probability=True)
}

# =====================================
# TRAIN MODELS
# =====================================

results = []

best_model = None
best_accuracy = 0

for name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(
        y_test,
        predictions,
        pos_label="Hire"
    )

    recall = recall_score(
        y_test,
        predictions,
        pos_label="Hire"
    )

    f1 = f1_score(
        y_test,
        predictions,
        pos_label="Hire"
    )

    cm = confusion_matrix(y_test, predictions)

    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1
    ])

    print("\n" + "="*50)
    print(name)
    print("="*50)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")

    print("\nConfusion Matrix")
    print(cm)

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model = pipeline

# =====================================
# RESULTS TABLE
# =====================================

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

print("\n\nMODEL COMPARISON")
print(results_df.sort_values(
    by="Accuracy",
    ascending=False
))

# =====================================
# SAVE BEST MODEL
# =====================================

joblib.dump(best_model, "model.pkl")

print("\nBest Model Saved Successfully!")
print("Filename: model.pkl")

Dataset Shape: (1000, 11)
   Resume_ID              Name                                        Skills  \
0          1        Ashley Ali                      TensorFlow, NLP, Pytorch   
1          2      Wesley Roman  Deep Learning, Machine Learning, Python, SQL   
2          3     Corey Sanchez         Ethical Hacking, Cybersecurity, Linux   
3          4  Elizabeth Carney                   Python, Pytorch, TensorFlow   
4          5        Julie Hill                              SQL, React, Java   

   Experience (Years) Education                Certifications  \
0                  10      B.Sc                           NaN   
1                  10       MBA                     Google ML   
2                   1       MBA  Deep Learning Specialization   
3                   7    B.Tech                 AWS Certified   
4                   4       PhD                           NaN   

                Job Role Recruiter Decision  Salary Expectation ($)  \
0          AI Researcher       

In [25]:
print(X.columns.tolist())

['Skills', 'Experience (Years)', 'Education', 'Certifications', 'Job Role', 'Salary Expectation ($)', 'Projects Count', 'AI Score (0-100)']


In [4]:
import pandas as pd
import joblib

# Load trained model
model = joblib.load("model.pkl")

# Sample candidate

candidate = {
    "Skills": ["Python,Machine Learning"],
    "Experience (Years)": [3],
    "Education": ["B.Tech"],
    "Certifications": ["AWS"],
    "Job Role": ["Data Scientist"],
    "Salary Expectation ($)": [70000],
    "Projects Count": [5],
    "AI Score (0-100)": [85]
}

df = pd.DataFrame(candidate)

prediction = model.predict(df)

probability = model.predict_proba(df)

print("Prediction:", prediction[0])

print(
    "Selection Probability:",
    round(max(probability[0]) * 100, 2),
    "%"
)

Prediction: Hire
Selection Probability: 100.0 %


In [5]:
import os

os.makedirs("backend", exist_ok=True)
os.makedirs("saved_model", exist_ok=True)

In [6]:
print(X.columns)

Index(['Skills', 'Experience (Years)', 'Education', 'Certifications',
       'Job Role', 'Salary Expectation ($)', 'Projects Count',
       'AI Score (0-100)'],
      dtype='object')


In [7]:
sample = {
    "Experience (Years)": 3,
    "AI Score (0-100)": 85,
    "Salary Expectation ($)": 50000,
    "Job Role": "Data Scientist",
    "Education": "B.Tech",
    "Projects Count": 4,
    "Certifications": "AWS",
    "Skills": "Python,Machine Learning"
}

In [8]:
def predict_candidate(data: dict):
    df = pd.DataFrame([data])

    # Reorder columns EXACTLY like training
    df = df.reindex(model.feature_names_in_, axis=1)

    prediction = model.predict(df)[0]
    probability = model.predict_proba(df)[0]

    return {
        "prediction": prediction,
        "confidence": float(max(probability))
    }

In [9]:
joblib.dump(pipeline, "model.pkl")

['model.pkl']

In [10]:
import joblib
import pandas as pd

model = joblib.load("model.pkl")

def predict_candidate(data: dict):

    # ✅ FIXED: correct 2D input
    df = pd.DataFrame([data])

    prediction = model.predict(df)[0]

    probability = model.predict_proba(df)[0]

    return {
        "prediction": prediction,
        "confidence": float(max(probability))
    }


# test
sample = {
    "Experience (Years)": 3,
    "AI Score (0-100)": 85,
    "Salary Expectation ($)": 50000,
    "Job Role": "Data Scientist",
    "Education": "B.Tech",
    "Projects Count": 4,
    "Certifications": "AWS",
    "Skills": "Python,ML"
}

print(predict_candidate(sample))

{'prediction': 'Hire', 'confidence': 0.9999584368866061}


In [11]:
pipeline.predict(df)

array(['Hire'], dtype=object)

In [12]:
joblib.dump(best_model, "model.pkl")

['model.pkl']

In [17]:
import os
import joblib

BASE_DIR = os.getcwd()   # ✅ current folder
MODEL_PATH = os.path.join(BASE_DIR, "model.pkl")

model = joblib.load(MODEL_PATH)

In [18]:
import os
print(os.getcwd())

C:\Users\siris


In [23]:
joblib.dump(best_model, "model.pkl")

['model.pkl']

In [24]:
joblib.dump(best_model, "model.pkl")

['model.pkl']

In [ ]:
import pandas as pd
import joblib

# Load the trained model pipeline
model_path = "model.pkl"
try:
    model = joblib.load(model_path)
    print("✅ Model loaded successfully!")
except Exception as e:
    print(f"❌ Error loading model: {e}")

def get_candidate_input():
    print("\n--- Enter Candidate Details for Screening ---")
    
    # 1. Skills
    skills = input("Skills (comma-separated, e.g., 'Python, Machine Learning'): ").strip()
    
    # 2. Experience (Years)
    while True:
        try:
            experience = float(input("Experience in Years (e.g., 3): "))
            break
        except ValueError:
            print("Please enter a valid number for experience.")
            
    # 3. Education
    education = input("Education Level (e.g., 'B.Tech', 'MBA', 'PhD'): ").strip()
    
    # 4. Certifications
    certs = input("Certifications (e.g., 'AWS', 'Google ML', or leave blank): ").strip()
    if not certs:
        certs = "Not Available"
        
    # 5. Job Role
    job_role = input("Job Role (e.g., 'Data Scientist', 'Software Engineer'): ").strip()
    
    # 6. Salary Expectation
    while True:
        try:
            salary = float(input("Salary Expectation in $ (e.g., 70000): "))
            break
        except ValueError:
            print("Please enter a valid number for salary.")
            
    # 7. Projects Count
    while True:
        try:
            projects = int(input("Number of Projects Completed (e.g., 5): "))
            break
        except ValueError:
            print("Please enter a valid integer for projects count.")
            
    # 8. AI Score
    while True:
        try:
            ai_score = float(input("AI Score 0-100 (e.g., 85): "))
            if 0 <= ai_score <= 100:
                break
            print("AI Score must be between 0 and 100.")
        except ValueError:
            print("Please enter a valid number for AI Score.")
            
    # Structure features exactly matching model expectations
    candidate_data = {
        "Skills": skills,
        "Experience (Years)": experience,
        "Education": education,
        "Certifications": certs,
        "Job Role": job_role,
        "Salary Expectation ($)": salary,
        "Projects Count": projects,
        "AI Score (0-100)": ai_score
    }
    
    return pd.DataFrame([candidate_data])

# Run interactive screening
try:
    df_candidate = get_candidate_input()
    
    # Predict using the pipeline
    prediction = model.predict(df_candidate)[0]
    probability = model.predict_proba(df_candidate)[0]
    confidence = max(probability) * 100
    
    print("\n" + "="*40)
    print("             SCREENING RESULT             ")
    print("="*40)
    if prediction == "Hire":
        print(f"STATUS    : ✅ HIRE")
    else:
        print(f"STATUS    : ❌ REJECT")
    print(f"CONFIDENCE: {confidence:.2f}%")
    print("="*40)
except Exception as e:
    print(f"An error occurred during prediction: {e}")